# Spark SQL

**SQL:** Structured Query Languaje

Spark 1.6 introdujo una nueva abstracción de programación llamad API estructurada.

Con los datos estructurados y esta forma las piezas de información Spark puede realizar optimizaciones para acelerar las aplicaciones de procesamiento de datos.

De esta manera los datos tienen un esquema y generan un archivo con un determinado formato accecible para spark SQL.

In [ ]:
!pip install findspark

## Crear un DataFrame con Spark

In [ ]:
import findspark
findspark.init()
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()
sc = spark.sparkContext

In [ ]:
rdd = sc.parallelize([item for item in range(10)]).map(lambda x: (x, x**2))
rdd.collect()

[(0, 0),
 (1, 1),
 (2, 4),
 (3, 9),
 (4, 16),
 (5, 25),
 (6, 36),
 (7, 49),
 (8, 64),
 (9, 81)]

In [ ]:
df = rdd.toDF(['numero', 'cuadrado'])

In [ ]:
df.printSchema()

root
 |-- numero: long (nullable = true)
 |-- cuadrado: long (nullable = true)



In [ ]:
df.show()

+------+--------+
|numero|cuadrado|
+------+--------+
|     0|       0|
|     1|       1|
|     2|       4|
|     3|       9|
|     4|      16|
|     5|      25|
|     6|      36|
|     7|      49|
|     8|      64|
|     9|      81|
+------+--------+



## Crear DataFrame a partir de un RDD con Schema

Hay dos formas

In [ ]:
rrd1 = sc.parallelize([(1, 'Jose', 20), (2, 'Juana', 30), (3, 'Marcos', 50)])

In [ ]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType

In [ ]:
# Primera forma:

esquema1 = StructType(
    [
        StructField('id', IntegerType(), True),
        StructField('nombre', StringType(), True),
        StructField('edad', IntegerType(), True)
    ]
)

In [ ]:
# Segunda forma:

esquema2 = "`id` INT, `nombre` STRING, `saldo` INT"


In [ ]:
df1 = spark.createDataFrame(rrd1, schema=esquema1)

In [ ]:
df1.printSchema()

root
 |-- id: integer (nullable = true)
 |-- nombre: string (nullable = true)
 |-- edad: integer (nullable = true)



In [ ]:
df1.show()

+---+------+----+
| id|nombre|edad|
+---+------+----+
|  1|  Jose|  20|
|  2| Juana|  30|
|  3|Marcos|  50|
+---+------+----+



In [ ]:
df2 = spark.createDataFrame(rrd1, schema=esquema2)

In [ ]:
df2.printSchema()

root
 |-- id: integer (nullable = true)
 |-- nombre: string (nullable = true)
 |-- saldo: integer (nullable = true)



In [ ]:
df2.show()

+---+------+-----+
| id|nombre|saldo|
+---+------+-----+
|  1|  Jose|   20|
|  2| Juana|   30|
|  3|Marcos|   50|
+---+------+-----+



## Crear un DataDrame a partir de datos externos

Las dos clases principales de Spark SQL son **DataFrameReader** y **DataFrameWriter**.

La primera la podremos invocar con **spark.read()**, la segunda se verá luego.

In [ ]:
import findspark
findspark.init()
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

### Archivo de texto

In [ ]:
# DataFrame mediante la lectura de un archivo de texto

df = spark.read.text('/content/drive/MyDrive/RDD_FILES/sparksql/dataTXT.txt')

df.show(truncate=False)

+-----------------------------------------------------------------------+
|value                                                                  |
+-----------------------------------------------------------------------+
|Estamos en el curso de pyspark                                         |
|En este capítulo estamos estudiando el API SQL de Saprk                |
|En esta sección estamos creado dataframes a partir de fuentes de datos,|
|y en este ejemplo creamos un dataframe a partir de un texto plano      |
+-----------------------------------------------------------------------+



### Archivo CSV

In [ ]:
df1 = spark.read.csv('/content/drive/MyDrive/RDD_FILES/sparksql/dataCSV.csv', header=True)
df1.show(truncate=False)

+-----------+-------------+--------------------------------------------------------------------------------------+---------------------+-----------+------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------+------+--------+-------------+----------------------------------------------+-----------------+----------------+----------------------+-------------------------------------------------------------------------------------------------------------------------------------------------

### Archivo CSV con delimitador

In [ ]:
df2 = spark.read.option('header', 'true').option('delimiter', '|').csv('/content/drive/MyDrive/RDD_FILES/sparksql/dataTab.txt')
df2.show()

+----+----+----------+-----+
|pais|edad|     fecha|color|
+----+----+----------+-----+
|  MX|  23|2021-02-21| rojo|
|  CA|  56|2021-06-10| azul|
|  US|  32|2020-06-02|verde|
+----+----+----------+-----+



### Archivo JSON

Proporcionaremos un esquema al JSON indicando los tipos de datos a utilizar

In [ ]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType

json_schema = StructType(
    [
        StructField('color', StringType(), True),
        StructField('edad', IntegerType(), True),
        StructField('fecha', DateType(), True),
        StructField('pais', StringType(), True)
    ]
)

In [ ]:
df4 = spark.read.schema(json_schema).json('/content/drive/MyDrive/RDD_FILES/sparksql/dataJSON.json')
df4.show()

+-----+----+----------+----+
|color|edad|     fecha|pais|
+-----+----+----------+----+
| rojo|NULL|2021-02-21|  MX|
| azul|NULL|2021-06-10|  CA|
|verde|NULL|2020-06-02|  US|
+-----+----+----------+----+



In [ ]:
df4.printSchema()

root
 |-- color: string (nullable = true)
 |-- edad: integer (nullable = true)
 |-- fecha: date (nullable = true)
 |-- pais: string (nullable = true)



### Archivo PARQUET

In [ ]:
df5 = spark.read.parquet('/content/drive/MyDrive/RDD_FILES/sparksql/dataPARQUET.parquet')
df5.show()

+-----------+-------------+--------------------+--------------------+-----------+--------------------+--------------------+-------+------+--------+-------------+--------------------+-----------------+----------------+----------------------+--------------------+
|   video_id|trending_date|               title|       channel_title|category_id|        publish_time|                tags|  views| likes|dislikes|comment_count|      thumbnail_link|comments_disabled|ratings_disabled|video_error_or_removed|         description|
+-----------+-------------+--------------------+--------------------+-----------+--------------------+--------------------+-------+------+--------+-------------+--------------------+-----------------+----------------+----------------------+--------------------+
|2kyS6SvSYSE|     17.14.11|WE WANT TO TALK A...|        CaseyNeistat|         22|2017-11-13T17:13:...|     SHANtell martin| 748374| 57527|    2966|        15954|https://i.ytimg.c...|            False|           Fal

In [ ]:
df6 = spark.read.format('parquet').load('/content/drive/MyDrive/RDD_FILES/sparksql/dataPARQUET.parquet')
df6.show()

+-----------+-------------+--------------------+--------------------+-----------+--------------------+--------------------+-------+------+--------+-------------+--------------------+-----------------+----------------+----------------------+--------------------+
|   video_id|trending_date|               title|       channel_title|category_id|        publish_time|                tags|  views| likes|dislikes|comment_count|      thumbnail_link|comments_disabled|ratings_disabled|video_error_or_removed|         description|
+-----------+-------------+--------------------+--------------------+-----------+--------------------+--------------------+-------+------+--------+-------------+--------------------+-----------------+----------------+----------------------+--------------------+
|2kyS6SvSYSE|     17.14.11|WE WANT TO TALK A...|        CaseyNeistat|         22|2017-11-13T17:13:...|     SHANtell martin| 748374| 57527|    2966|        15954|https://i.ytimg.c...|            False|           Fal

## Trabajo con columnas

Las operaciones estructuradas reflejan el tipo de operaciones que se pueden hacer con SQL.

Estas se dividen en:

- **Transformación**
- **Acción**

In [ ]:
import findspark
findspark.init()
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

### Ver que columnas tenemos

In [ ]:
df = spark.read.parquet('/content/drive/MyDrive/RDD_FILES/sparksql/dataPARQUET.parquet')
df.printSchema()

root
 |-- video_id: string (nullable = true)
 |-- trending_date: string (nullable = true)
 |-- title: string (nullable = true)
 |-- channel_title: string (nullable = true)
 |-- category_id: string (nullable = true)
 |-- publish_time: string (nullable = true)
 |-- tags: string (nullable = true)
 |-- views: string (nullable = true)
 |-- likes: string (nullable = true)
 |-- dislikes: string (nullable = true)
 |-- comment_count: string (nullable = true)
 |-- thumbnail_link: string (nullable = true)
 |-- comments_disabled: string (nullable = true)
 |-- ratings_disabled: string (nullable = true)
 |-- video_error_or_removed: string (nullable = true)
 |-- description: string (nullable = true)



### select

In [ ]:
df.select('title').show(truncate=False)

+--------------------------------------------------------------------------------------+
|title                                                                                 |
+--------------------------------------------------------------------------------------+
|WE WANT TO TALK ABOUT OUR MARRIAGE                                                    |
|The Trump Presidency: Last Week Tonight with John Oliver (HBO)                        |
|Racist Superman | Rudy Mancuso, King Bach & Lele Pons                                 |
|Nickelback Lyrics: Real or Fake?                                                      |
|I Dare You: GOING BALD!?                                                              |
|2 Weeks with iPhone X                                                                 |
|Roy Moore & Jeff Sessions Cold Open - SNL                                             |
|5 Ice Cream Gadgets put to the Test                                                   |
|The Greatest Showman

In [ ]:
from pyspark.sql.functions import col

df.select(col('title')).show(truncate=False)

+--------------------------------------------------------------------------------------+
|title                                                                                 |
+--------------------------------------------------------------------------------------+
|WE WANT TO TALK ABOUT OUR MARRIAGE                                                    |
|The Trump Presidency: Last Week Tonight with John Oliver (HBO)                        |
|Racist Superman | Rudy Mancuso, King Bach & Lele Pons                                 |
|Nickelback Lyrics: Real or Fake?                                                      |
|I Dare You: GOING BALD!?                                                              |
|2 Weeks with iPhone X                                                                 |
|Roy Moore & Jeff Sessions Cold Open - SNL                                             |
|5 Ice Cream Gadgets put to the Test                                                   |
|The Greatest Showman

### SELECTEXPR

In [ ]:
import findspark
findspark.init()
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

In [ ]:
from pyspark.sql.functions import col

df.select(col('video_id')).show()

+-----------+
|   video_id|
+-----------+
|2kyS6SvSYSE|
|1ZAPwfrtAFY|
|5qpjK5DgCt4|
|puqaWrEC7tY|
|d380meD0W0M|
|gHZ1Qz0KiKM|
|39idVpFF7NQ|
|nc99ccSXST0|
|jr9QtXwC9vc|
|TUmyygCMMGA|
|9wRQljFNDW8|
|VifQlJit6A0|
|5E4ZBSInqUU|
|GgVmn66oK_A|
|TaTleo4cOs8|
|kgaO45SyaO4|
|ZAQs-ctOqXQ|
|YVfyYrEmzgM|
|eNSN6qet1kE|
|B5HORANmzHw|
+-----------+
only showing top 20 rows


In [ ]:
df.select('video_id', 'trending_date').show()

+-----------+-------------+
|   video_id|trending_date|
+-----------+-------------+
|2kyS6SvSYSE|     17.14.11|
|1ZAPwfrtAFY|     17.14.11|
|5qpjK5DgCt4|     17.14.11|
|puqaWrEC7tY|     17.14.11|
|d380meD0W0M|     17.14.11|
|gHZ1Qz0KiKM|     17.14.11|
|39idVpFF7NQ|     17.14.11|
|nc99ccSXST0|     17.14.11|
|jr9QtXwC9vc|     17.14.11|
|TUmyygCMMGA|     17.14.11|
|9wRQljFNDW8|     17.14.11|
|VifQlJit6A0|     17.14.11|
|5E4ZBSInqUU|     17.14.11|
|GgVmn66oK_A|     17.14.11|
|TaTleo4cOs8|     17.14.11|
|kgaO45SyaO4|     17.14.11|
|ZAQs-ctOqXQ|     17.14.11|
|YVfyYrEmzgM|     17.14.11|
|eNSN6qet1kE|     17.14.11|
|B5HORANmzHw|     17.14.11|
+-----------+-------------+
only showing top 20 rows


In [ ]:
from pyspark.sql.functions import col
from pyspark.sql.types import IntegerType

df.select(
    col('likes').cast(IntegerType()),
    col('dislikes').cast(IntegerType()),
    (col('likes').cast(IntegerType()) - col('dislikes').alias('aceptacion'))
).show()

+------+--------+---------------------------------------------+
| likes|dislikes|(CAST(likes AS INT) - dislikes AS aceptacion)|
+------+--------+---------------------------------------------+
| 57527|    2966|                                        54561|
| 97185|    6146|                                        91039|
|146033|    5339|                                       140694|
| 10172|     666|                                         9506|
|132235|    1989|                                       130246|
|  9763|     511|                                         9252|
| 15993|    2445|                                        13548|
| 23663|     778|                                        22885|
|  3543|     119|                                         3424|
| 12654|    1363|                                        11291|
|   655|      25|                                          630|
|  1576|     303|                                         1273|
|114188|    1333|                       

In [ ]:
df.selectExpr('CAST(likes AS INT) as likes', 'CAST(dislikes AS INT) as dislikes', '(CAST(likes AS INT) - CAST(dislikes AS INT)) as aceptacion').show()

+------+--------+----------+
| likes|dislikes|aceptacion|
+------+--------+----------+
| 57527|    2966|     54561|
| 97185|    6146|     91039|
|146033|    5339|    140694|
| 10172|     666|      9506|
|132235|    1989|    130246|
|  9763|     511|      9252|
| 15993|    2445|     13548|
| 23663|     778|     22885|
|  3543|     119|      3424|
| 12654|    1363|     11291|
|   655|      25|       630|
|  1576|     303|      1273|
|114188|    1333|    112855|
|  7848|    1171|      6677|
|  7473|     246|      7227|
|  9419|      52|      9367|
|  8011|     638|      7373|
|  5398|      53|      5345|
| 11963|      36|     11927|
|  8421|     191|      8230|
+------+--------+----------+
only showing top 20 rows


In [ ]:
df.selectExpr("count(distinct(video_id)) as videos").show()

+------+
|videos|
+------+
|  6837|
+------+



### filter y where

Con `df.filter()` podrémos filtrar según la condición indicada en el argumento. `where` hace lo mismo. Podremos tener condiciones de todo tipo con `and` y `or`.


In [ ]:
import findspark
findspark.init()
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

In [ ]:
df = spark.read.parquet('/content/drive/MyDrive/RDD_FILES/sparksql/dataPARQUET.parquet')

In [ ]:
df.show()

+-----------+-------------+--------------------+--------------------+-----------+--------------------+--------------------+-------+------+--------+-------------+--------------------+-----------------+----------------+----------------------+--------------------+
|   video_id|trending_date|               title|       channel_title|category_id|        publish_time|                tags|  views| likes|dislikes|comment_count|      thumbnail_link|comments_disabled|ratings_disabled|video_error_or_removed|         description|
+-----------+-------------+--------------------+--------------------+-----------+--------------------+--------------------+-------+------+--------+-------------+--------------------+-----------------+----------------+----------------------+--------------------+
|2kyS6SvSYSE|     17.14.11|WE WANT TO TALK A...|        CaseyNeistat|         22|2017-11-13T17:13:...|     SHANtell martin| 748374| 57527|    2966|        15954|https://i.ytimg.c...|            False|           Fal

In [ ]:
from pyspark.sql.functions import col

df.filter(col('video_id') == '2kyS6SvSYSE').show()

+-----------+-------------+--------------------+-------------+-----------+--------------------+---------------+-------+-----+--------+-------------+--------------------+-----------------+----------------+----------------------+--------------------+
|   video_id|trending_date|               title|channel_title|category_id|        publish_time|           tags|  views|likes|dislikes|comment_count|      thumbnail_link|comments_disabled|ratings_disabled|video_error_or_removed|         description|
+-----------+-------------+--------------------+-------------+-----------+--------------------+---------------+-------+-----+--------+-------------+--------------------+-----------------+----------------+----------------------+--------------------+
|2kyS6SvSYSE|     17.14.11|WE WANT TO TALK A...| CaseyNeistat|         22|2017-11-13T17:13:...|SHANtell martin| 748374|57527|    2966|        15954|https://i.ytimg.c...|            False|           False|                 False|SHANTELL'S CHANNE...|
|2ky

In [ ]:
df.where(col('trending_date') == '17.14.11').show()

+-----------+-------------+--------------------+--------------------+-----------+--------------------+--------------------+-------+------+--------+-------------+--------------------+-----------------+----------------+----------------------+--------------------+
|   video_id|trending_date|               title|       channel_title|category_id|        publish_time|                tags|  views| likes|dislikes|comment_count|      thumbnail_link|comments_disabled|ratings_disabled|video_error_or_removed|         description|
+-----------+-------------+--------------------+--------------------+-----------+--------------------+--------------------+-------+------+--------+-------------+--------------------+-----------------+----------------+----------------------+--------------------+
|2kyS6SvSYSE|     17.14.11|WE WANT TO TALK A...|        CaseyNeistat|         22|2017-11-13T17:13:...|     SHANtell martin| 748374| 57527|    2966|        15954|https://i.ytimg.c...|            False|           Fal

### distinct y dropDuplicates

Eliminan los duplicados

In [ ]:
import findspark
findspark.init()
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

In [ ]:
df = spark.read.parquet('/content/drive/MyDrive/RDD_FILES/sparksql/dataPARQUET.parquet')

In [ ]:
# Distinct
df_sin_duplicados = df.distinct()

In [ ]:
print(f"El dataframe original tiene: {df.count()} filas")
print(f"El dataframe sin duplicados tiene: {df_sin_duplicados.count()} filas")

El dataframe original tiene: 48137 filas
El dataframe sin duplicados tiene: 41497 filas


In [ ]:
# dropDuplicates: elimina duplicados considerando el primer registro como valido, o el de índice menor

dataframe = spark.createDataFrame([(1, 'azul', 567), (2, 'rojo', 487), (1, 'azul', 345), (2, 'verde', 783)]).toDF('id', 'color', 'importe')

dataframe.show()

dataframe.dropDuplicates(['id', 'color']).show()

+---+-----+-------+
| id|color|importe|
+---+-----+-------+
|  1| azul|    567|
|  2| rojo|    487|
|  1| azul|    345|
|  2|verde|    783|
+---+-----+-------+

+---+-----+-------+
| id|color|importe|
+---+-----+-------+
|  1| azul|    567|
|  2| rojo|    487|
|  2|verde|    783|
+---+-----+-------+



### sort y orderBy

In [ ]:
import findspark
findspark.init()
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

In [ ]:
from pyspark.sql.functions import col

df = (spark.read.parquet('/content/drive/MyDrive/RDD_FILES/sparksql/dataPARQUET.parquet')
    .select(col('likes'), col('views'), col('video_id'), col('dislikes'))
    .dropDuplicates(['video_id'])
)

df.show()

+-----+-------+-----------+--------+
|likes|  views|   video_id|dislikes|
+-----+-------+-----------+--------+
| 6531| 475965|-0CMnp02rNY|     172|
| 4429| 563746|-0NYY8cqdiQ|      54|
|32752|1566807|-1Hm41N0dUs|     393|
| 5214| 129360|-1yT-K3c6YI|     108|
|  438|  67429|-2RVw2_QyxQ|      23|
|19339|1012527|-2aVkGcI7ZA|     633|
| 1444|  84744|-2b4qSoMnKE|     199|
|10350| 703371|-2wRFv-mScQ|     260|
|73480| 545655|-35jibKqbEo|     727|
|    2|   2863|-37nIo_tLnk|       0|
| 4028| 385104|-39ysKKpE7I|     343|
| 6468| 230360|-3h4Xt9No9o|     177|
|10384| 249601|-3nEHRN6IPg|     370|
|38776| 296237|-4s2MeUgduo|     466|
|71090| 390631|-5aaJJQFvOg|     635|
|21224| 744363|-66xHRJSPxs|     534|
|17882| 363370|-7AZX5Xtiks|     416|
|36960| 908989|-7UzyXO-mzk|     434|
|17120|1815030|-7_ATlZ-zMc|     633|
|  760| 252542|-8ZHXaGILlc|     100|
+-----+-------+-----------+--------+
only showing top 20 rows


In [ ]:
from pyspark.sql.functions import desc

df.sort(desc('likes')).show()

+-----+-------+-----------+--------+
|likes|  views|   video_id|dislikes|
+-----+-------+-----------+--------+
|99990|2079137|2v4-L4PkV9U|    2844|
|99973|2465294|DSRSgMp5X1w|   17299|
|99952|3313449|LdhQzXHYLZ4|    5142|
| 9991|  98513|eBnXbImHX-g|      91|
| 9988|1162843|kz1xzBYppW8|    2555|
|99851|1053828|vRf3azp1pak|    1226|
| 9984| 206669|Lydh_saD9EQ|      88|
| 9984| 254807|Ps7GzIV2KP0|     294|
|  998|  71308|Hkx5fveyjIs|      74|
|  998|  54348|Pr6zjrF0Djg|      75|
|  998|  82087|hX643KbiI4s|      93|
|99761|1454233|h5CLO2n6OxQ|     692|
|  997|  27234|nb42DxagyOE|      13|
| 9969| 273905|c47kn_Y4y8A|     127|
| 9946| 242329|QXcbVHFE2bo|     148|
| 9939| 235293|1iGBHh1q0Kg|     232|
| 9926| 467558|hHFuZVGpBC0|     342|
|99254|1552618|0v-6AylRH68|    5195|
| 9925| 166235|flLc6LmAG6c|      50|
| 9921| 594536|e9NOwaiXqqA|     323|
+-----+-------+-----------+--------+
only showing top 20 rows


In [ ]:
df.orderBy(col('views').desc()).show()

+------+-------+-----------+--------+
| likes|  views|   video_id|dislikes|
+------+-------+-----------+--------+
|126363| 999910|gw82GrEt370|    1034|
| 78088| 999867|cyhU06cXfeU|     690|
| 58552| 998908|QIN5_tJRiyY|    1080|
|151348|9988608|fAIX12F6958|   10274|
| 70972| 998362|LC3fWTXZXxE|    1608|
|  4727|  99796|kOsl3cmK3zg|     152|
|   120|   9977|1L_fPteZOYQ|      11|
|   299|  99674|Yzx_tSlifIw|      95|
|119634| 996318|__1SjDrSMik|    1143|
|  3959|  99619|9ymjcSvEyhk|     158|
|   969|  99577|qLVZiKDNxtw|     197|
|  2136|  99566|pWZnhJay0Y8|     419|
|  6487|  99535|BfbIoUMdKZ0|     116|
|  3761|  99510|bhqH6tTr_Lk|     615|
|   258| 994747|i7FgneNlM14|     398|
| 21094| 994662|-CS84oCtjvc|     714|
|107029| 993913|EWf7P3okX9s|     472|
|  9507| 993593|vYb4_ARPNfo|    3647|
|  7388| 993435|z1FfOwjlqxU|     919|
|   111|   9932|b3KFfgoDzw8|       9|
+------+-------+-----------+--------+
only showing top 20 rows


In [ ]:
# Se pueden emular practicamente todas las sentencias SQL

top10 = df.orderBy(col('views').desc()).limit(10)
top10.show()

+------+-------+-----------+--------+
| likes|  views|   video_id|dislikes|
+------+-------+-----------+--------+
|126363| 999910|gw82GrEt370|    1034|
| 78088| 999867|cyhU06cXfeU|     690|
| 58552| 998908|QIN5_tJRiyY|    1080|
|151348|9988608|fAIX12F6958|   10274|
| 70972| 998362|LC3fWTXZXxE|    1608|
|  4727|  99796|kOsl3cmK3zg|     152|
|   120|   9977|1L_fPteZOYQ|      11|
|   299|  99674|Yzx_tSlifIw|      95|
|119634| 996318|__1SjDrSMik|    1143|
|  3959|  99619|9ymjcSvEyhk|     158|
+------+-------+-----------+--------+



### withColumn y withColumnRenamed

Creamos una nueva columna con datos previos o nuevos según indiquemos.

In [ ]:
import findspark
findspark.init()
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

In [ ]:
df = spark.read.parquet('/content/drive/MyDrive/RDD_FILES/sparksql/dataPARQUET.parquet')

In [ ]:
from pyspark.sql.functions import col
from pyspark.sql.types import IntegerType

df_valoracion = df.withColumn(
    'valoracion',
    col('likes').cast(IntegerType()) - col('dislikes').cast(IntegerType())
)

In [ ]:
df_valoracion.printSchema()

root
 |-- video_id: string (nullable = true)
 |-- trending_date: string (nullable = true)
 |-- title: string (nullable = true)
 |-- channel_title: string (nullable = true)
 |-- category_id: string (nullable = true)
 |-- publish_time: string (nullable = true)
 |-- tags: string (nullable = true)
 |-- views: string (nullable = true)
 |-- likes: string (nullable = true)
 |-- dislikes: string (nullable = true)
 |-- comment_count: string (nullable = true)
 |-- thumbnail_link: string (nullable = true)
 |-- comments_disabled: string (nullable = true)
 |-- ratings_disabled: string (nullable = true)
 |-- video_error_or_removed: string (nullable = true)
 |-- description: string (nullable = true)
 |-- valoracion: integer (nullable = true)



In [ ]:
df_valoracion1 = (
    df.withColumn('valoracion', col('likes').cast(IntegerType()) - col('dislikes').cast(IntegerType()))
    .withColumn('res_div', col('valoracion') % 10)
)

df_valoracion1.printSchema()

root
 |-- video_id: string (nullable = true)
 |-- trending_date: string (nullable = true)
 |-- title: string (nullable = true)
 |-- channel_title: string (nullable = true)
 |-- category_id: string (nullable = true)
 |-- publish_time: string (nullable = true)
 |-- tags: string (nullable = true)
 |-- views: string (nullable = true)
 |-- likes: string (nullable = true)
 |-- dislikes: string (nullable = true)
 |-- comment_count: string (nullable = true)
 |-- thumbnail_link: string (nullable = true)
 |-- comments_disabled: string (nullable = true)
 |-- ratings_disabled: string (nullable = true)
 |-- video_error_or_removed: string (nullable = true)
 |-- description: string (nullable = true)
 |-- valoracion: integer (nullable = true)
 |-- res_div: integer (nullable = true)



In [ ]:
df_renombrado = df.withColumnRenamed('video_id', 'id') # Prestar atención al log anterior

df_renombrado.printSchema()

root
 |-- id: string (nullable = true)
 |-- trending_date: string (nullable = true)
 |-- title: string (nullable = true)
 |-- channel_title: string (nullable = true)
 |-- category_id: string (nullable = true)
 |-- publish_time: string (nullable = true)
 |-- tags: string (nullable = true)
 |-- views: string (nullable = true)
 |-- likes: string (nullable = true)
 |-- dislikes: string (nullable = true)
 |-- comment_count: string (nullable = true)
 |-- thumbnail_link: string (nullable = true)
 |-- comments_disabled: string (nullable = true)
 |-- ratings_disabled: string (nullable = true)
 |-- video_error_or_removed: string (nullable = true)
 |-- description: string (nullable = true)



### drop, sample y randomsplit

**drop**: borra atributos, se pueden asignar uno o más
**sample**: retorna una serie de filas aleatorias del dataset

In [ ]:
import findspark
findspark.init()
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

In [ ]:
df = spark.read.parquet('/content/drive/MyDrive/RDD_FILES/sparksql/dataPARQUET.parquet')

In [ ]:
df.printSchema()

root
 |-- video_id: string (nullable = true)
 |-- trending_date: string (nullable = true)
 |-- title: string (nullable = true)
 |-- channel_title: string (nullable = true)
 |-- category_id: string (nullable = true)
 |-- publish_time: string (nullable = true)
 |-- tags: string (nullable = true)
 |-- views: string (nullable = true)
 |-- likes: string (nullable = true)
 |-- dislikes: string (nullable = true)
 |-- comment_count: string (nullable = true)
 |-- thumbnail_link: string (nullable = true)
 |-- comments_disabled: string (nullable = true)
 |-- ratings_disabled: string (nullable = true)
 |-- video_error_or_removed: string (nullable = true)
 |-- description: string (nullable = true)



In [ ]:
df_util = df.drop('comments_disabled')

In [ ]:
df_util.printSchema()

root
 |-- video_id: string (nullable = true)
 |-- trending_date: string (nullable = true)
 |-- title: string (nullable = true)
 |-- channel_title: string (nullable = true)
 |-- category_id: string (nullable = true)
 |-- publish_time: string (nullable = true)
 |-- tags: string (nullable = true)
 |-- views: string (nullable = true)
 |-- likes: string (nullable = true)
 |-- dislikes: string (nullable = true)
 |-- comment_count: string (nullable = true)
 |-- thumbnail_link: string (nullable = true)
 |-- ratings_disabled: string (nullable = true)
 |-- video_error_or_removed: string (nullable = true)
 |-- description: string (nullable = true)



In [ ]:
df_muestra = df.sample(fraction=0.8)
df_muestra.show()

+-----------+-------------+--------------------+--------------------+-----------+--------------------+--------------------+-------+------+--------+-------------+--------------------+-----------------+----------------+----------------------+--------------------+
|   video_id|trending_date|               title|       channel_title|category_id|        publish_time|                tags|  views| likes|dislikes|comment_count|      thumbnail_link|comments_disabled|ratings_disabled|video_error_or_removed|         description|
+-----------+-------------+--------------------+--------------------+-----------+--------------------+--------------------+-------+------+--------+-------------+--------------------+-----------------+----------------+----------------------+--------------------+
|2kyS6SvSYSE|     17.14.11|WE WANT TO TALK A...|        CaseyNeistat|         22|2017-11-13T17:13:...|     SHANtell martin| 748374| 57527|    2966|        15954|https://i.ytimg.c...|            False|           Fal

In [ ]:
num_filas_total = df.count() - (df.count()*0.2)
num_filas_muestra = df_muestra.count()

num_filas_total, num_filas_muestra

(38509.6, 38529)

In [ ]:
# Podremos a la fracción anterior proporcionarla una semilla de aleatoriedad

df_muestra_seed = df.sample(fraction=0.8, seed=42)
df_muestra_seed.show()

+-----------+-------------+--------------------+--------------------+-----------+--------------------+--------------------+-------+------+--------+-------------+--------------------+-----------------+----------------+----------------------+--------------------+
|   video_id|trending_date|               title|       channel_title|category_id|        publish_time|                tags|  views| likes|dislikes|comment_count|      thumbnail_link|comments_disabled|ratings_disabled|video_error_or_removed|         description|
+-----------+-------------+--------------------+--------------------+-----------+--------------------+--------------------+-------+------+--------+-------------+--------------------+-----------------+----------------+----------------------+--------------------+
|2kyS6SvSYSE|     17.14.11|WE WANT TO TALK A...|        CaseyNeistat|         22|2017-11-13T17:13:...|     SHANtell martin| 748374| 57527|    2966|        15954|https://i.ytimg.c...|            False|           Fal

In [ ]:
# Trabajo con withReplacement genera un subconjunto aleatorio de un DataFrame donde las filas pueden repetirse

df_muestra = df.sample(withReplacement=True, fraction=0.2)
df_muestra.show()

+--------------------+-------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------+------+--------+-------------+--------------------+-----------------+----------------+----------------------+--------------------+
|            video_id|trending_date|               title|       channel_title|         category_id|        publish_time|                tags|   views| likes|dislikes|comment_count|      thumbnail_link|comments_disabled|ratings_disabled|video_error_or_removed|         description|
+--------------------+-------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------+------+--------+-------------+--------------------+-----------------+----------------+----------------------+--------------------+
|         puqaWrEC7tY|     17.14.11|Nickelback Lyrics...|Good Mythical Mor...|                  24|2017-11-13T11:00:...|"rhett and link"|...|  343168| 10172|

In [ ]:
# randomSplit: parecido al train_test_split

df_train, df_test = df.randomSplit([0.8, 0.2], seed=12345)

In [ ]:
df_train.show()

+-----------+-------------+--------------------+--------------------+-----------+--------------------+--------------------+-------+-----+--------+-------------+--------------------+-----------------+----------------+----------------------+--------------------+
|   video_id|trending_date|               title|       channel_title|category_id|        publish_time|                tags|  views|likes|dislikes|comment_count|      thumbnail_link|comments_disabled|ratings_disabled|video_error_or_removed|         description|
+-----------+-------------+--------------------+--------------------+-----------+--------------------+--------------------+-------+-----+--------+-------------+--------------------+-----------------+----------------+----------------------+--------------------+
|-0CMnp02rNY|     18.06.06|Mindy Kaling's Da...|        TheEllenShow|         24|2018-06-04T13:00:...|"ellen"|"ellen de...| 475965| 6531|     172|          271|https://i.ytimg.c...|            False|           False| 

In [ ]:
df_test.show()

+-----------+-------------+--------------------+--------------------+-----------+--------------------+--------------------+--------+------+--------+-------------+--------------------+-----------------+----------------+----------------------+--------------------+
|   video_id|trending_date|               title|       channel_title|category_id|        publish_time|                tags|   views| likes|dislikes|comment_count|      thumbnail_link|comments_disabled|ratings_disabled|video_error_or_removed|         description|
+-----------+-------------+--------------------+--------------------+-----------+--------------------+--------------------+--------+------+--------+-------------+--------------------+-----------------+----------------+----------------------+--------------------+
|-0NYY8cqdiQ|     18.01.02|Megan Mullally Di...|        TheEllenShow|         24|2018-01-29T14:00:...|"megan mullally"|...|  563746|  4429|      54|           94|https://i.ytimg.c...|            False|          

## Trabajo con datos incorrectos o faltantes

In [ ]:
import findspark
findspark.init()
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

In [ ]:
df = spark.read.parquet('/content/drive/MyDrive/RDD_FILES/sparksql/dataPARQUET.parquet')

In [ ]:
df.count()

48137

In [ ]:
df.na.drop().count() # Esto se puede hacer de muchas formas diferentes

40390

In [ ]:
df.na.drop(subset=['views']).count()

41061

## Acciones sobre un datadrame sparkSQL.

In [ ]:
import findspark
findspark.init()
from pyspark.sql import SparkSession


In [ ]:
df = spark.read.parquet('/content/drive/MyDrive/RDD_FILES/sparksql/dataPARQUET.parquet')

In [ ]:
df.show(5) # Primeros elementos indicados

+-----------+-------------+--------------------+--------------------+-----------+--------------------+--------------------+-------+------+--------+-------------+--------------------+-----------------+----------------+----------------------+--------------------+
|   video_id|trending_date|               title|       channel_title|category_id|        publish_time|                tags|  views| likes|dislikes|comment_count|      thumbnail_link|comments_disabled|ratings_disabled|video_error_or_removed|         description|
+-----------+-------------+--------------------+--------------------+-----------+--------------------+--------------------+-------+------+--------+-------------+--------------------+-----------------+----------------+----------------------+--------------------+
|2kyS6SvSYSE|     17.14.11|WE WANT TO TALK A...|        CaseyNeistat|         22|2017-11-13T17:13:...|     SHANtell martin| 748374| 57527|    2966|        15954|https://i.ytimg.c...|            False|           Fal

In [ ]:
df.take(2) # Elemento indicado

[Row(video_id='2kyS6SvSYSE', trending_date='17.14.11', title='WE WANT TO TALK ABOUT OUR MARRIAGE', channel_title='CaseyNeistat', category_id='22', publish_time='2017-11-13T17:13:01.000Z', tags='SHANtell martin', views='748374', likes='57527', dislikes='2966', comment_count='15954', thumbnail_link='https://i.ytimg.com/vi/2kyS6SvSYSE/default.jpg', comments_disabled='False', ratings_disabled='False', video_error_or_removed='False', description="SHANTELL'S CHANNEL - https://www.youtube.com/shantellmartin\\nCANDICE - https://www.lovebilly.com\\n\\nfilmed this video in 4k on this -- http://amzn.to/2sTDnRZ\\nwith this lens -- http://amzn.to/2rUJOmD\\nbig drone - http://tinyurl.com/h4ft3oy\\nOTHER GEAR ---  http://amzn.to/2o3GLX5\\nSony CAMERA http://amzn.to/2nOBmnv\\nOLD CAMERA; http://amzn.to/2o2cQBT\\nMAIN LENS; http://amzn.to/2od5gBJ\\nBIG SONY CAMERA; http://amzn.to/2nrdJRO\\nBIG Canon CAMERA; http://tinyurl.com/jn4q4vz\\nBENDY TRIPOD THING; http://tinyurl.com/gw3ylz2\\nYOU NEED THIS FOR 

In [ ]:
# df.head(2) # Primeros elementos de cabecera

In [ ]:
# df.select('likes').collect()